# Model Performance Analysis

Classifier evaluation (accuracy / precision / recall / F1 / ROC-AUC / PR-AUC / confusion matrix), forecasting evaluation (cosine similarity / gene correlation / DEG delta panel), and classifier biological interpretation (logit perturbation impact / pathway enrichment).

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
import scanpy as sc
from sklearn.metrics import classification_report, precision_recall_fscore_support

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config
from model.utils.reproducibility import seed_everything
from model.utils.device import get_device
from model.utils.constants import get_pv_path_nodes, get_cluster_column
from model.models import Classifier, ClassifierConfig, Forecaster, ForecasterConfig
from model.training.checkpointing import load_checkpoint
from model.data.preprocessing import prepare_classifier_data, prepare_clusters
from model.data.trajectory_pairs import PrepareTrajectoryData, TrajectoryDataset
from model.analysis import (
    evaluate_classifier_binary,
    evaluate_classifier,
    compute_cosine_similarity,
    compute_expression_correlation,
    compute_transition_deg_delta_panel,
    compute_forecasting_accuracy,
    compute_forecasting_manifold_metrics,
    compute_source_shuffle_diagnostic,
    compute_target_bin_residual_diagnostic,
    compute_transition_dynamic_gene_evaluation,
    compute_umap_projection,
    fit_predict_ridge_baseline,
    get_forecaster_predictions,
    make_source_shuffled_data_list,
    predict_target_bin_mean_baseline,
    GlobalLogitImpactEngine,
    EnrichmentConfig,
    PathwayEnrichmentRunner,
)

DEVICE = get_device()
print('DEVICE:', DEVICE)


In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> fuller evaluation
SEED = 42
seed_everything(SEED)

# Classifier evaluation — must match training split exactly
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15            # test = 1 - 0.70 - 0.15 = 0.15
CLS_BATCH_SIZE = 64

# Forecasting evaluation
MAX_ITEMS = 100 if SMOKE_MODE else None   # None = all held-out pairs; smoke uses seeded subset
GEN_BATCH_SIZE = 32 if SMOKE_MODE else 128
UMAP_MAX_POINTS = 100 if SMOKE_MODE else 2000
DYNAMIC_TOP_N = 50
DYNAMIC_MIN_PAIRS = 3

OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'performance'
OUTDIR.mkdir(parents=True, exist_ok=True)

CLS_CFG = load_yaml_config(ROOT / 'configs' / 'classifier_binary.yaml')
GEN_CFG = load_yaml_config(ROOT / 'configs' / 'forecasting.yaml')

adata_path = ROOT / CLS_CFG['data']['h5ad_path']
cls_ckpt_path = ROOT / CLS_CFG['training']['checkpoint_path']
gen_ckpt_path = ROOT / GEN_CFG['training']['checkpoint_path']

print('adata_path:', adata_path)
print('cls_ckpt_path:', cls_ckpt_path)
print('gen_ckpt_path:', gen_ckpt_path)
print('SMOKE_MODE:', SMOKE_MODE)

# Biological interpretation (classifier logit impact + enrichment)
RUN_ENRICHMENT = True        # requires gseapy + network access for Enrichr
N_IMPACT_REPEATS = 2 if SMOKE_MODE else 10
N_PER_CLASS = 80 if SMOKE_MODE else 300


## 1) Classifier Performance

In [ ]:
# Prepare classifier data (binary PV vs NPV)
cls_dataset, cls_stats = prepare_classifier_data(
    h5ad_path=str(adata_path),
    max_len=int(CLS_CFG['data']['max_len']),
    exclude_class=CLS_CFG['data'].get('exclude_class'),
    label_mode='binary',
    verbose=True,
)

# Rebuild the exact same held-out split used during training.
# Must use the same ratios and generator seed as training.
n_total = len(cls_dataset)
train_size = int(TRAIN_RATIO * n_total)
val_size = int(VAL_RATIO * n_total)
test_size = n_total - train_size - val_size

generator = torch.Generator().manual_seed(SEED)
_, _, test_set = random_split(cls_dataset, [train_size, val_size, test_size], generator=generator)
eval_loader = DataLoader(test_set, batch_size=CLS_BATCH_SIZE, shuffle=False)
print(f'Held-out test set restored: {len(test_set)} cells, batches: {len(eval_loader)}')

In [4]:
# Load classifier and evaluate
model_cfg = dict(CLS_CFG['model'])
if model_cfg.get('vocab_size') in [None, 'null']:
    model_cfg['vocab_size'] = int(cls_stats['vocab_size'])

classifier = Classifier(ClassifierConfig.from_dict(model_cfg)).to(DEVICE)
classifier, cls_ckpt = load_checkpoint(classifier, cls_ckpt_path, device=DEVICE)
classifier.eval()
print('Classifier loaded. checkpoint keys:', list(cls_ckpt.keys())[:8])

# Run binary evaluation
cls_result = evaluate_classifier_binary(classifier, eval_loader, DEVICE)
metrics = cls_result['metrics']

print('\n--- Classifier Binary Metrics ---')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print('\nConfusion matrix:')
print(cls_result['confusion_matrix'])
print('\nClassification report:')
print(cls_result['classification_report'])

Classifier loaded. checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'metrics']

--- Classifier Binary Metrics ---
  n_samples: 1898
  accuracy: 0.9821
  precision: 0.9772
  recall: 0.9713
  f1: 0.9742
  roc_auc: 0.9980
  pr_auc: 0.9969

Confusion matrix:
[[1221   15]
 [  19  643]]

Classification report:
              precision    recall  f1-score   support

      NPV(0)       0.98      0.99      0.99      1236
       PV(1)       0.98      0.97      0.97       662

    accuracy                           0.98      1898
   macro avg       0.98      0.98      0.98      1898
weighted avg       0.98      0.98      0.98      1898



In [5]:
# Save classifier metrics
with open(OUTDIR / 'classifier_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved:', OUTDIR / 'classifier_metrics.json')

cm_df = pd.DataFrame(
    cls_result['confusion_matrix'],
    index=['True_NPV', 'True_PV'],
    columns=['Pred_NPV', 'Pred_PV'],
)
cm_df.to_csv(OUTDIR / 'classifier_confusion_matrix.csv')
print('Saved:', OUTDIR / 'classifier_confusion_matrix.csv')
display(cm_df)

Saved: F:\Study\final\model\artifacts\notebooks\performance\classifier_metrics.json
Saved: F:\Study\final\model\artifacts\notebooks\performance\classifier_confusion_matrix.csv


,Pred_NPV,Pred_PV
True_NPV,1221,15
True_PV,19,643


## 1b) Multi-Class Classifier Performance

In [ ]:
# Multi-class classifier evaluation (7 PV+NPV clusters)
MULTI_CLS_CFG = load_yaml_config(ROOT / 'configs' / 'classifier_multi.yaml')
multi_ckpt_path = ROOT / MULTI_CLS_CFG['training']['checkpoint_path']
print('Multi-class config loaded. checkpoint:', multi_ckpt_path)

# Prepare multi-class data
multi_dataset, multi_stats = prepare_classifier_data(
    h5ad_path=str(adata_path),
    max_len=int(MULTI_CLS_CFG['data']['max_len']),
    exclude_class=MULTI_CLS_CFG['data'].get('exclude_class'),
    label_mode='multi',
    cluster_col=MULTI_CLS_CFG['data'].get('cluster_col'),
    verbose=True,
)

# Rebuild the exact same held-out split used during training
n_total_m = len(multi_dataset)
train_size_m = int(TRAIN_RATIO * n_total_m)
val_size_m = int(VAL_RATIO * n_total_m)
test_size_m = n_total_m - train_size_m - val_size_m

generator_m = torch.Generator().manual_seed(SEED)
_, _, test_set_m = random_split(multi_dataset, [train_size_m, val_size_m, test_size_m], generator=generator_m)
eval_loader_m = DataLoader(test_set_m, batch_size=CLS_BATCH_SIZE, shuffle=False)
print(f'Multi-class held-out test set restored: {len(test_set_m)} cells, batches: {len(eval_loader_m)}')

# Load multi-class classifier (peek checkpoint for num_classes to avoid shape mismatch)
multi_cfg_dict = dict(MULTI_CLS_CFG['model'])
if multi_cfg_dict.get('vocab_size') in [None, 'null']:
    multi_cfg_dict['vocab_size'] = int(multi_stats['vocab_size'])
if multi_cfg_dict.get('num_classes') in [None, 'null']:
    _ckpt = torch.load(multi_ckpt_path, map_location='cpu', weights_only=True)
    for _k, _v in _ckpt['model_state_dict'].items():
        if 'classifier' in _k and 'weight' in _k and _v.ndim == 2:
            multi_cfg_dict['num_classes'] = int(_v.shape[0])
            break
    else:
        multi_cfg_dict['num_classes'] = multi_stats['num_classes']
    del _ckpt

multi_cls = Classifier(ClassifierConfig.from_dict(multi_cfg_dict)).to(DEVICE)
multi_cls, multi_ckpt = load_checkpoint(multi_cls, multi_ckpt_path, device=DEVICE)
multi_cls.eval()
print(f'Multi-class classifier loaded: {multi_cfg_dict["num_classes"]} classes')

# Evaluate
multi_result = evaluate_classifier(multi_cls, eval_loader_m, DEVICE, num_classes=multi_cfg_dict['num_classes'])
print(f'\nMulti-class accuracy: {multi_result["accuracy"]:.4f}')
print(f'Per-class accuracy: {multi_result["per_class_accuracy"]}')

# Per-class precision/recall/f1
y_true_m = multi_result['all_labels'].numpy()
y_pred_m = multi_result['all_preds'].numpy()
class_names = multi_stats['class_names']

per_class = precision_recall_fscore_support(
    y_true_m, y_pred_m, labels=range(len(class_names)), zero_division=0
)
multi_metrics_df = pd.DataFrame({
    'class_index': range(len(class_names)),
    'cluster': class_names,
    'precision': per_class[0],
    'recall': per_class[1],
    'f1': per_class[2],
    'support': per_class[3],
})
multi_metrics_df = multi_metrics_df.sort_values('recall')
print('\nPer-class metrics (sorted by recall, ascending):')
display(multi_metrics_df)


In [ ]:
# Confusion matrix with class labels
cm = multi_result['confusion_matrix']
cm_df = pd.DataFrame(
    cm,
    index=[f'True_{c}' for c in class_names],
    columns=[f'Pred_{c}' for c in class_names],
)
print('Multi-class confusion matrix:')
display(cm_df)

# Identify low-recall classes (matching demo spotlight on 7.1, 16.1)
low_recall = multi_metrics_df[multi_metrics_df['recall'] < 0.8]
if not low_recall.empty:
    print(f'\nLow recall classes (recall < 0.8):')
    for _, row in low_recall.iterrows():
        print(f'  Cluster {row["cluster"]}: recall={row["recall"]:.4f}, f1={row["f1"]:.4f}, support={int(row["support"])}')

# Save multi-class metrics
multi_metrics_df.to_csv(OUTDIR / 'classifier_multi_metrics.csv', index=False)
print('Saved:', OUTDIR / 'classifier_multi_metrics.csv')


### 1c) Biological Interpretation — Logit Impact

Perturbation-based gene importance via decision-logit perturbation. Each gene's
impact score reflects how much zeroing that gene's expression shifts the
PV-vs-NPV decision boundary. Higher MeanAbsDelta = greater influence on classification.

In [ ]:
# Run global logit impact analysis on the classifier
gli_engine = GlobalLogitImpactEngine(
    model=classifier,
    adata=adata,
    device=DEVICE,
    max_len=int(CLS_CFG['data']['max_len']),
    output_dir=str(OUTDIR / 'LogitImpactResults'),
)
impact_summary_df, impact_repeat_df, impact_meta_df = gli_engine.run_repeated_subsampling(
    n_repeats=N_IMPACT_REPEATS,
    n_per_class=N_PER_CLASS,
    replace=False,
    random_state=SEED,
    save_prefix='global_logit_impact',
    verbose=True,
)

print(f'\nTop 20 genes by global logit impact:')
display(impact_summary_df.head(20))
impact_summary_df.to_csv(OUTDIR / 'global_logit_impact_summary.csv', index=False)
print('Saved:', OUTDIR / 'global_logit_impact_summary.csv')

### 1d) Pathway Enrichment on Impact Genes

Reactome pathway over-representation on top logit-impact genes (ranked by MeanAbsDelta).

In [ ]:
if RUN_ENRICHMENT:
    runner = PathwayEnrichmentRunner(
        config=EnrichmentConfig(
            output_dir=str(OUTDIR / 'EnrichmentResults'),
            verbose=True,
        )
    )
    enrich_logit = runner.run_from_df(
        df=impact_summary_df,
        gene_col='Gene',
        score_col='MeanAbsDelta',
        top_n=80 if SMOKE_MODE else 150,
        min_freq_col='MeanFrequency',
        min_freq=0.05,
        title='Reactome enrichment: global logit-impact genes',
    )
    display(enrich_logit.get('sig_results', pd.DataFrame()).head(20))
    runner.save_results(
        enrich_res=enrich_logit,
        filename='reactome_logit_sig_results.csv',
    )
    print('Saved enrichment results to:', str(OUTDIR / 'EnrichmentResults'))
else:
    print('RUN_ENRICHMENT=False -> skipped')

## 2) Forecasting Performance

In [ ]:
# Prepare trajectory data with source-disjoint train/val/held-out splits
adata = sc.read_h5ad(adata_path)
adata = prepare_clusters(adata, GEN_CFG['data'].get('cluster_col'))
print(adata)

prep = PrepareTrajectoryData(
    h5ad_path=str(adata_path),
    config={'model_params': GEN_CFG['model']},
    subset_col=GEN_CFG['data'].get('subset_col', 'trajectory_class'),
    subset_values=tuple(GEN_CFG['data'].get('subset_values', ['PV'])),
    time_col=GEN_CFG['data'].get('time_col'),
    cluster_col=GEN_CFG['data'].get('cluster_col'),
    n_bins=int(GEN_CFG['data'].get('n_bins', 120)),
    allowed_offsets=tuple(GEN_CFG['data'].get('allowed_offsets', [1, 2, 3, 4, 5, 6])),
    base_max_dist=float(GEN_CFG['data'].get('base_max_dist', 12.0)),
    dist_alpha=float(GEN_CFG['data'].get('dist_alpha', 1.0)),
    allowed_cross_steps=tuple(GEN_CFG['data'].get('allowed_cross_steps', [1, 2])),
    k_intra=int(GEN_CFG['data'].get('k_intra', 1)),
    k_cross=int(GEN_CFG['data'].get('k_cross', 2)),
    val_split=float(GEN_CFG['data'].get('val_split', 0.2)),
    heldout_split=float(GEN_CFG['data'].get('heldout_split', 0.1)),
    pair_diagnostics=True,
)

train_data_list = prep.train_data
val_data_list = prep.val_data
heldout_data_list = prep.heldout_data

if len(heldout_data_list) == 0:
    raise ValueError('Held-out dataset is empty. Increase data size or reduce heldout_split.')
if 'time_bin' not in prep.adata.obs.columns:
    raise KeyError("PrepareTrajectoryData must populate prep.adata.obs['time_bin'] for reproducible diagnostics.")

if MAX_ITEMS is None:
    eval_indices = np.arange(len(heldout_data_list))
else:
    rng = np.random.default_rng(SEED)
    eval_indices = np.sort(rng.choice(len(heldout_data_list), size=min(MAX_ITEMS, len(heldout_data_list)), replace=False))

eval_data_list = [heldout_data_list[int(i)] for i in eval_indices]
heldout_ds = TrajectoryDataset(eval_data_list)
heldout_loader = DataLoader(heldout_ds, batch_size=GEN_BATCH_SIZE, shuffle=False)

print(f'Train pairs: {len(train_data_list)} | Val pairs: {len(val_data_list)} | Held-out total: {len(heldout_data_list)}')
print(f'Evaluating held-out pairs: {len(eval_data_list)}')


In [ ]:
# Load forecasting model
f_model = Forecaster(ForecasterConfig.from_dict(GEN_CFG['model'])).to(DEVICE)
f_model, f_ckpt = load_checkpoint(f_model, gen_ckpt_path, device=DEVICE)
f_model.eval()
print('Forecaster loaded. checkpoint keys:', list(f_ckpt.keys())[:8])

In [ ]:
# Inference and reproducibility baselines on the same held-out pairs
print('Generating Transformer predictions on held-out pairs ...')
preds_np, trues_np, nz_masks, meta = get_forecaster_predictions(f_model, heldout_loader, DEVICE)
source_expr = meta['source_val']

print('Generating source-shuffle Transformer predictions within target bins ...')
shuffled_source_data_list, source_shuffle_info = make_source_shuffled_data_list(
    eval_data_list=eval_data_list,
    adata=prep.adata,
    time_bin_col='time_bin',
    seed=SEED,
)
source_shuffle_loader = DataLoader(TrajectoryDataset(shuffled_source_data_list), batch_size=GEN_BATCH_SIZE, shuffle=False)
source_shuffle_preds, _, _, _ = get_forecaster_predictions(f_model, source_shuffle_loader, DEVICE)

print('Generating deterministic baselines on the same held-out pairs ...')
target_bin_mean_preds, target_bin_mean_info = predict_target_bin_mean_baseline(
    train_data_list=train_data_list,
    eval_data_list=eval_data_list,
    adata=prep.adata,
    time_bin_col='time_bin',
)
ridge_preds, ridge_info = fit_predict_ridge_baseline(
    train_data_list=train_data_list,
    val_data_list=val_data_list,
    eval_data_list=eval_data_list,
)

baseline_predictions = {
    'Target-bin mean': target_bin_mean_preds,
    f"Ridge (alpha={ridge_info['alpha']:g})": ridge_preds,
}
baseline_details = {
    'Target-bin mean': target_bin_mean_info,
    'Ridge': ridge_info,
    'Source shuffle': source_shuffle_info,
}

preds = torch.from_numpy(preds_np)
trues = torch.from_numpy(trues_np)
n_items = len(eval_data_list)

print('preds shape:', preds_np.shape, 'trues shape:', trues_np.shape)
print(f"Target-bin mean bins: {target_bin_mean_info['n_bins']} | fallbacks: {target_bin_mean_info['fallback_count']}")
print(f"Source-shuffle pairs: {source_shuffle_info['n_pairs']} | singleton/self donors: {source_shuffle_info['singleton_count']}")
print(f"Ridge alpha selected: {ridge_info['alpha']:g} | validation MSE: {ridge_info['validation_mse']:.6f}")


In [ ]:
# Global accuracy and baseline comparison
cos_sim = compute_cosine_similarity(preds, trues)
cos_sim_np = cos_sim.numpy()
cos_summary = {
    'cosine_mean': float(np.mean(cos_sim_np)),
    'cosine_median': float(np.median(cos_sim_np)),
    'cosine_std': float(np.std(cos_sim_np)),
    'cosine_q25': float(np.percentile(cos_sim_np, 25)),
    'cosine_q75': float(np.percentile(cos_sim_np, 75)),
    'n_pairs': int(len(cos_sim_np)),
}
print('Cosine similarity summary:')
for k, v in cos_summary.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

target_indices = np.asarray(meta['target_idx']).astype(int)
accuracy_rows = []
for model_name, model_preds in {'Transformer': preds_np, **baseline_predictions}.items():
    row = compute_forecasting_accuracy(model_preds, trues_np, nz_masks, target_indices, prep.adata)
    row['Model'] = model_name
    accuracy_rows.append(row)
accuracy_df = pd.DataFrame(accuracy_rows)[['Model', 'mse', 'global_corr', 'meta_corr', 'n_meta_bins']]
print('\nGlobal accuracy vs baselines:')
display(accuracy_df)


In [ ]:
# Compute per-gene expression correlation
from model.analysis import compute_expression_correlation

gene_corr = compute_expression_correlation(preds, trues)
gene_corr_nz = gene_corr[gene_corr != 0]
corr_summary = {
    'gene_corr_mean': float(np.mean(gene_corr_nz)) if len(gene_corr_nz) > 0 else float('nan'),
    'gene_corr_median': float(np.median(gene_corr_nz)) if len(gene_corr_nz) > 0 else float('nan'),
    'gene_corr_std': float(np.std(gene_corr_nz)) if len(gene_corr_nz) > 0 else float('nan'),
    'n_genes_nonzero_corr': int(np.sum(gene_corr != 0)),
    'n_genes_total': int(len(gene_corr)),
}
print('Gene correlation summary:')
for k, v in corr_summary.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


In [ ]:
# Transition dynamic gene (DVG) and residual diagnostics
cluster_key = GEN_CFG['data'].get('cluster_col') or get_cluster_column()

dynamic_gene_df = compute_transition_dynamic_gene_evaluation(
    preds=preds_np,
    trues=trues_np,
    source_expr=source_expr,
    meta=meta,
    adata=prep.adata,
    baseline_preds=baseline_predictions,
    cluster_key=cluster_key,
    top_n=DYNAMIC_TOP_N,
    min_pairs=DYNAMIC_MIN_PAIRS,
)
print(f'Transition dynamic gene evaluation (top_n={DYNAMIC_TOP_N}, min_pairs={DYNAMIC_MIN_PAIRS}):')
if dynamic_gene_df.empty:
    print('No cluster transitions had enough held-out pairs for this diagnostic.')
else:
    display(dynamic_gene_df.sort_values(['Transition', 'Model']))

residual_df = compute_target_bin_residual_diagnostic(
    preds=preds_np,
    trues=trues_np,
    target_bin_mean_preds=target_bin_mean_preds,
    baseline_preds=baseline_predictions,
)
source_shuffle_df = compute_source_shuffle_diagnostic(
    preds=preds_np,
    source_shuffle_preds=source_shuffle_preds,
    trues=trues_np,
    target_bin_mean_preds=target_bin_mean_preds,
)
print('\nResidual after target-bin mean:')
display(residual_df)
print('\nSource-shuffle residual diagnostic:')
display(source_shuffle_df)

# Backward-compatible DEG delta panel across predefined PV path transitions
trajectory_nodes = get_pv_path_nodes()
deg_df = compute_transition_deg_delta_panel(
    preds=preds_np,
    trues=trues_np,
    meta=meta,
    adata=prep.adata,
    trajectory_nodes=trajectory_nodes,
    cluster_col=cluster_key,
    top_n=50,
)
if not deg_df.empty:
    deg_summary = (
        deg_df.groupby(['source_cluster', 'target_cluster'], observed=True)
        .agg(pearson_r=('pearson_r', 'first'), pearson_p=('pearson_p', 'first'), valid_pairs=('valid_count', 'first'))
        .reset_index()
    )
    deg_summary['transition'] = deg_summary['source_cluster'].astype(str) + '->' + deg_summary['target_cluster'].astype(str)
    display(deg_summary)
else:
    deg_summary = pd.DataFrame()
    print('No predefined trajectory DEG delta pairs found.')


In [ ]:
# Pair-level geometry: UMAP projection and manifold conservation metrics
# batch_key can be provided for batch mixing ASW (e.g. batch_key="sample_id")
manifold_metrics = compute_forecasting_manifold_metrics(
    preds=preds_np,
    target_indices=target_indices,
    adata=prep.adata,
    cluster_key=cluster_key,
    seed=SEED,
)
print('Manifold conservation metrics:')
for k, v in manifold_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

try:
    umap_true, umap_pred = compute_umap_projection(
        preds=preds_np,
        trues=trues_np,
        seed=SEED,
        max_points=UMAP_MAX_POINTS,
        save_path=OUTDIR / 'true_reference_umap.pkl',
    )
    umap_df = pd.concat([
        pd.DataFrame({'UMAP1': umap_true[:, 0], 'UMAP2': umap_true[:, 1], 'kind': 'true'}),
        pd.DataFrame({'UMAP1': umap_pred[:, 0], 'UMAP2': umap_pred[:, 1], 'kind': 'pred'}),
    ], ignore_index=True)
    display(umap_df.head())
    print('Saved fitted UMAP reducer:', OUTDIR / 'true_reference_umap.pkl')
except ImportError as exc:
    umap_true = umap_pred = None
    umap_df = pd.DataFrame()
    print('UMAP skipped because umap-learn is unavailable:', exc)


In [ ]:
# Save forecasting metrics and diagnostic tables
forecasting_metrics = {
    'cosine_summary': cos_summary,
    'accuracy_vs_baselines': accuracy_df.to_dict(orient='records'),
    'gene_correlation_summary': corr_summary,
    'dynamic_gene_summary': dynamic_gene_df.to_dict(orient='records') if not dynamic_gene_df.empty else [],
    'residual_summary': residual_df.to_dict(orient='records'),
    'source_shuffle_summary': source_shuffle_df.to_dict(orient='records'),
    'manifold_metrics': manifold_metrics,
    'deg_delta_transition_summary': deg_summary.to_dict(orient='records') if not deg_summary.empty else [],
    'baseline_details': {
        'Target-bin mean': target_bin_mean_info,
        'Ridge': ridge_info,
        'Source shuffle': {k: v for k, v in source_shuffle_info.items() if k != 'donor_indices'},
    },
    'n_heldout_items_evaluated': int(n_items),
    'n_heldout_items_total': int(len(heldout_data_list)),
    'eval_indices_seed': int(SEED),
}
with open(OUTDIR / 'forecasting_metrics.json', 'w') as f:
    json.dump(forecasting_metrics, f, indent=2)
print('Saved:', OUTDIR / 'forecasting_metrics.json')

accuracy_df.to_csv(OUTDIR / 'forecasting_accuracy_vs_baselines.csv', index=False)
dynamic_gene_df.to_csv(OUTDIR / 'dynamic_gene_evaluation.csv', index=False)
residual_df.to_csv(OUTDIR / 'target_bin_residual_diagnostic.csv', index=False)
source_shuffle_df.to_csv(OUTDIR / 'source_shuffle_diagnostic.csv', index=False)
deg_df.to_csv(OUTDIR / 'deg_delta_panel.csv', index=False)
if not umap_df.empty:
    umap_df.to_csv(OUTDIR / 'umap_true_pred_projection.csv', index=False)
print('Saved forecasting diagnostic tables to:', OUTDIR)

print('\n=== All performance outputs saved to', OUTDIR, '===')
